In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Optional


In [ ]:
# Helper

def export_df_to_excel(
    df: pd.DataFrame,
    file_path: str,
    sheet_name: str = "data",
    index: bool = True,
):
    df.to_excel(
        file_path,
        sheet_name=sheet_name,
        index=index
    )


# 1 - Importer Indice historique


# Black&Schole Model


### main

In [ ]:

@dataclass
class BSParams:
    r_annual_cc: float
    sigma_annual: float
    s0: float
    s0_date: Optional[pd.Timestamp]

# =========================
# 2) Load index levels from Excel (Col A date, Col B level)
# =========================
def load_index_series_AB(
    xlsx_path: str,
    sheet_name: Optional[str] = None,
) -> pd.Series:
    """
    Đọc time series level của indice từ Excel:
      - Cột A (col 0): Date (từ hàng 2)
      - Cột B (col 1): Index level (từ hàng 2)

    Output:
      pd.Series with DatetimeIndex (date) and float values (level), sorted asc.
    """
    df = pd.read_excel(
        xlsx_path,
        sheet_name=sheet_name if sheet_name is not None else 0,
        header=0,          # hàng 1 là header, data từ hàng 2
        usecols="A:B"      # chỉ lấy cột A,B
    )

    date_col = df.columns[0]
    level_col = df.columns[1]

    # parse date
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    # parse level (handle French formats: "1 234,56")
    lvl = (
        df[level_col]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    df[level_col] = pd.to_numeric(lvl, errors="coerce")

    s = (
        df[[date_col, level_col]]
        .dropna()
        .drop_duplicates(subset=date_col)
        .set_index(date_col)[level_col]
        .sort_index()
    )

    if s.empty:
        raise ValueError("Series rỗng sau khi đọc/clean. Check sheet_name và dữ liệu cột A,B.")

    if (s <= 0).any():
        raise ValueError("Có giá trị level <= 0, không phù hợp cho log-return/GBM.")

    return s


def estimate_sigma_from_history(levels: pd.Series, day_count: int = 365) -> float:
    """sigma_annual = std(log-return daily) * sqrt(day_count)"""
    s = levels.sort_index()
    logrets = np.log(s / s.shift(1)).dropna()
    return float(logrets.std(ddof=1) * np.sqrt(day_count))


def get_s0(
    levels: pd.Series,
    start_date: str,
    default_s0: float = 1000.0,
    use_real_if_available: bool = False,
) -> tuple[float, Optional[pd.Timestamp]]:
    """Chọn S0: default hoặc lấy đúng level tại start_date nếu có."""
    if not use_real_if_available:
        return float(default_s0), None

    d = pd.Timestamp(start_date)
    if d in levels.index:
        return float(levels.loc[d]), d
    return float(default_s0), None


def build_bs_params_simple(
    levels: pd.Series,
    start_date: str,
    r_neutre_annual_cc: float,
    day_count: int = 365,
    s0_default: float = 1000.0,
    use_real_s0: bool = False,
) -> BSParams:
    """
    Build params đơn giản:
      - sigma: từ dữ liệu quá khứ
      - r_cc: = r_neutre (bạn truyền vào)
      - s0: default hoặc lấy level thật tại start_date nếu có
    """
    s0, s0_date = get_s0(
        levels,
        start_date=start_date,
        default_s0=s0_default,
        use_real_if_available=use_real_s0,
    )

    sigma = estimate_sigma_from_history(levels, day_count=day_count)

    return BSParams(
        r_annual_cc=float(r_neutre_annual_cc),
        sigma_annual=float(sigma),
        s0=float(s0),
        s0_date=s0_date,
    )


def simulate_gbm_monthly(
    s0: float,
    r_annual_cc: float,
    sigma_annual: float,
    start_date: str,
    n_months: int,
    n_sims: int,
    seed: Optional[int] = 42,
) -> pd.DataFrame:
    """GBM monthly simulation, output scenario x dates."""
    rng = np.random.default_rng(seed)

    dt = 1.0 / 12.0
    dates = pd.date_range(start=pd.Timestamp(start_date), periods=n_months + 1, freq="M")

    Z = rng.standard_normal(size=(n_months, n_sims))
    drift = (r_annual_cc - 0.5 * sigma_annual**2) * dt
    diffusion = sigma_annual * np.sqrt(dt) * Z

    log_paths = np.vstack([np.zeros((1, n_sims)), np.cumsum(drift + diffusion, axis=0)])
    paths = s0 * np.exp(log_paths)

    df = pd.DataFrame(paths.T, index=np.arange(1, n_sims + 1), columns=dates)
    df.index.name = "scenario"
    return df


In [ ]:

# =========================
# 8) Example usage
# =========================
if __name__ == "__main__":
    xlsx = "Data.xlsx"

    # r_neutre_cc = 0.0259
    # r_neutre_cc =0.011135201286952185
    r_neutre_cc = 0.0691009521484375
    levels = load_index_series_AB(xlsx)
    taux_df = pd.read_excel("Data.xlsx",sheet_name="Taux")

    params = build_bs_params_simple(
        levels=levels,
        start_date="2025-12-31",
        r_neutre_annual_cc=r_neutre_cc,
        s0_default=1000,
        use_real_s0=False,
    )



    paths = simulate_gbm_monthly(
        s0=params.s0,
        r_annual_cc=params.r_annual_cc,
        sigma_annual=params.sigma_annual,
        start_date="2025-12-31",
        n_months=145,
        n_sims=10_000,
        seed=42,
    )


/tmp/ipykernel_367/1173245976.py:129: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(start=pd.Timestamp(start_date), periods=n_months + 1, freq="M")


In [ ]:
params

BSParams(r_annual_cc=0.0691009521484375, sigma_annual=0.2558122680177892, s0=1000.0, s0_date=None)

In [ ]:
paths

,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31,2026-09-30,...,2037-04-30,2037-05-31,2037-06-30,2037-07-31,2037-08-31,2037-09-30,2037-10-31,2037-11-30,2037-12-31,2038-01-31
scenario,,,,,,,,,,,,,,,,,,,,,
1,1000.0,1025.862869,1042.459994,925.114746,861.141640,808.229368,858.095952,911.913717,800.217678,842.627221,...,1642.443886,1507.752017,1241.873096,1234.873917,1124.128968,1167.567027,1230.313801,1240.421270,1281.642614,1335.572192
2,1000.0,928.887558,995.670813,1066.631773,1066.132287,974.488415,853.015882,836.162453,880.401587,926.794136,...,826.208929,681.791716,716.779543,743.528452,876.997491,862.351697,894.807172,843.672452,905.945257,888.622620
3,1000.0,1060.192027,953.858024,952.662903,865.120049,889.324893,876.377087,984.015377,1010.935414,1046.732103,...,1691.792905,1670.302191,1814.884122,1684.723878,1684.059339,1581.506986,1646.675346,1689.486630,1712.585885,1861.923651
4,1000.0,1075.181296,1055.962563,1013.270491,964.050429,977.989801,905.050892,842.599629,806.614721,916.698614,...,1332.570952,1183.549626,1061.053586,1006.255191,1213.310918,1225.104131,1268.187444,1365.050109,1227.213774,1133.330287
5,1000.0,868.449670,926.090136,911.855275,898.278655,764.510592,947.416480,839.364977,773.040655,753.542935,...,5784.848791,5351.751302,5319.292689,5690.921909,5638.402180,5893.701154,5826.585439,5083.332408,5091.506402,5073.299904
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9996,1000.0,1130.512586,1291.790929,1317.124774,1225.493203,1255.292620,1264.565442,1204.016500,1386.190182,1367.961694,...,5010.636993,4619.338681,5764.025889,5498.495979,5640.449237,6286.623857,6446.358260,6711.442411,6617.199293,5786.349884
9997,1000.0,998.367465,943.108105,959.355509,887.451229,1018.452893,1045.341940,1136.215493,1115.089490,1248.734253,...,6399.089964,6042.843076,5718.222196,5513.668091,6031.794215,6091.664139,5340.913671,4715.789382,4990.783502,5420.803552
9998,1000.0,1009.370718,1061.536702,1222.259735,1372.273292,1417.546747,1349.718619,1211.028950,1310.928176,1274.847105,...,9447.473383,8632.923082,8470.203690,8347.564981,8328.323702,6798.106483,6367.683121,6711.757108,6881.677838,6997.434974


Decrement

In [ ]:
import pandas as pd
import numpy as np
from typing import Literal

DecrementMode = Literal["flat", "cum"]

def decrement_paths(
    df: pd.DataFrame,
    decrement_value: float = 50.0,   # 50 points per year
    periods_per_year: int = 12,      # monthly
    mode: DecrementMode = "cum",     # với logic bạn nêu: dùng "cum"
) -> pd.DataFrame:
    """
    Linear decrement per period with cumulative option.

    Your target behavior (mode="cum", monthly):
      col0 (t0): 0
      col1:  -50/12
      col2:  -2*50/12
      ...
      colk:  -k*50/12

    mode:
      - "flat": subtract 50/12 mỗi kỳ (không tích lũy)
      - "cum" : subtract k*(50/12) (tích lũy theo số kỳ)
    """
    df_dec = df.copy()

    # ensure datetime columns + sort
    cols = pd.to_datetime(df_dec.columns)
    order = np.argsort(cols.values)
    df_dec = df_dec.iloc[:, order]

    n_cols = df_dec.shape[1]
    if n_cols == 0:
        return df_dec

    per_period = float(decrement_value) / float(periods_per_year)  # 50/12

    # k = 0,1,2,...,n_cols-1 where col0 is t0
    k = np.arange(n_cols, dtype=float)

    if mode == "flat":
        decrements = per_period * (k > 0)          # col0=0, các cột sau trừ 50/12
    elif mode == "cum":
        decrements = per_period * k                # col0=0, col1=1*50/12, col2=2*50/12,...
    else:
        raise ValueError("mode must be 'flat' or 'cum'")

    df_dec.iloc[:, :] = df_dec.values - decrements
    return df_dec


In [ ]:

path_decrement = decrement_paths(paths, decrement_value=50, mode = 'cum')


In [ ]:
path_decrement.head()

,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31,2026-09-30,...,2037-04-30,2037-05-31,2037-06-30,2037-07-31,2037-08-31,2037-09-30,2037-10-31,2037-11-30,2037-12-31,2038-01-31
scenario,,,,,,,,,,,,,,,,,,,,,
1,1000.0,1021.696203,1034.126660,912.614746,844.474973,787.396035,833.095952,882.747050,766.884344,805.127221,...,1075.777219,936.918683,666.873096,655.707250,540.795634,580.067027,638.647134,644.587937,681.642614,731.405525
2,1000.0,924.720891,987.337480,1054.131773,1049.465620,953.655082,828.015882,806.995786,847.068254,889.294136,...,259.542262,110.958383,141.779543,164.361785,293.664158,274.851697,303.140505,247.839118,305.945257,284.455953
3,1000.0,1056.025361,945.524691,940.162903,848.453382,868.491560,851.377087,954.848710,977.602081,1009.232103,...,1125.126238,1099.468857,1239.884122,1105.557211,1100.726005,994.006986,1055.008679,1093.653297,1112.585885,1257.756984
4,1000.0,1071.014629,1047.629230,1000.770491,947.383763,957.156468,880.050892,813.432963,773.281387,879.198614,...,765.904285,612.716293,486.053586,427.088524,629.977585,637.604131,676.520777,769.216775,627.213774,529.163620
5,1000.0,864.283003,917.756802,899.355275,881.611988,743.677259,922.416480,810.198310,739.707322,716.042935,...,5218.182124,4780.917968,4744.292689,5111.755243,5055.068847,5306.201154,5234.918772,4487.499075,4491.506402,4469.133238


In [ ]:
from typing import Literal
import pandas as pd
import numpy as np

FreqSortie = Literal["mensuelle", "trimestrielle", "annuelle"]

def _freq_to_months(freq_sortie: FreqSortie) -> int:
    if freq_sortie == "mensuelle":
        return 1
    if freq_sortie == "trimestrielle":
        return 3
    if freq_sortie == "annuelle":
        return 12
    raise ValueError("freq_sortie phải là 'mensuelle', 'trimestrielle', hoặc 'annuelle'.")

def Payoff(
    paths: pd.DataFrame,
    s_ref: float,  # mốc tham chiếu để xét barrières (ví dụ 1000)
    barriere_sortie_anticipe: float,
    barriere_sortie_maturite: float,
    barriere_protection: float,
    premiere_annee_sortie: int,
    freq_sortie: FreqSortie,
    coupon_par_periode: float,
    annee_finale: int,
    day_count: int = 365,
) -> pd.DataFrame:
    """
    Payoff SAF với ratio = S_t / S_ref.
    Output có thêm cột S_0 (giá tại date_initiale của mỗi scenario).
    """
    if paths.shape[1] < 2:
        raise ValueError("paths phải có ít nhất 2 cột ngày.")

    df = paths.copy()
    df.columns = pd.to_datetime(df.columns)
    df = df.loc[:, sorted(df.columns)]

    scenarios = df.index
    date_initiale = df.columns[0]
    pricing_date = date_initiale

    # <<< S_0 theo scenario (lấy từ paths) >>>
    S_0 = df.iloc[:, 0].astype(float)

    step_months = _freq_to_months(freq_sortie)

    start_obs_date = date_initiale + pd.DateOffset(years=premiere_annee_sortie)
    maturity_target = date_initiale + pd.DateOffset(years=annee_finale)

    available_dates = df.columns[df.columns <= maturity_target]
    if len(available_dates) == 0:
        raise ValueError("Không có dates nào <= maturity_target.")

    maturity_date = available_dates[-1]

    # obs_dates
    months_from_initial = ((available_dates.year - date_initiale.year) * 12 +
                           (available_dates.month - date_initiale.month))

    obs_dates, obs_months = [], []
    for dt, m in zip(available_dates, months_from_initial):
        if dt >= start_obs_date and (int(m) % step_months == 0):
            obs_dates.append(dt)
            obs_months.append(int(m))

    # init
    exit_date = pd.Series(maturity_date, index=scenarios, dtype="datetime64[ns]")
    nb_periodes = pd.Series(index=scenarios, dtype=int)
    called = pd.Series(False, index=scenarios)

    # maturity ratio vs S_ref
    sT = df[maturity_date].astype(float)
    rT = (sT / float(s_ref)).astype(float)

    # autocall
    if len(obs_dates) > 0:
        obs_period_counts = [m // step_months for m in obs_months]

        for d_obs, k_period in zip(obs_dates, obs_period_counts):
            if called.all():
                break

            st = df[d_obs].astype(float)
            ratio = (st / float(s_ref)).astype(float)

            hit = (ratio >= barriere_sortie_anticipe) & (~called)
            if hit.any():
                exit_date.loc[hit] = d_obs
                nb_periodes.loc[hit] = int(k_period)
                called.loc[hit] = True

    # not called
    not_called = ~called
    maturity_months = int((maturity_date.year - date_initiale.year) * 12 +
                          (maturity_date.month - date_initiale.month))
    maturity_periods = maturity_months // step_months
    nb_periodes.loc[not_called] = maturity_periods

    # payoff
    payoff = pd.Series(index=scenarios, dtype=float)

    payoff.loc[called] = float(s_ref) * (
        1.0 + coupon_par_periode * nb_periodes.loc[called].astype(float)
    )

    case1 = (rT >= barriere_sortie_maturite) & not_called
    payoff.loc[case1] = float(s_ref) * (
        1.0 + coupon_par_periode * nb_periodes.loc[case1].astype(float)
    )

    case2 = (rT >= barriere_protection) & (rT < barriere_sortie_maturite) & not_called
    payoff.loc[case2] = float(s_ref)

    case3 = (rT < barriere_protection) & not_called
    payoff.loc[case3] = float(s_ref) * rT.loc[case3].astype(float)

    # ratio_sortie (debug / analyse)
    ratio_sortie = pd.Series(index=scenarios, dtype=float)
    if len(obs_dates) > 0:
        for d_obs in obs_dates:
            mask = called & (exit_date == d_obs)
            if mask.any():
                ratio_sortie.loc[mask] = (
                    df.loc[mask, d_obs].astype(float) / float(s_ref)
                ).values
    ratio_sortie.loc[not_called] = rT.loc[not_called].values

    # T year fraction
    T = (pd.to_datetime(exit_date) - pd.to_datetime(pricing_date)).dt.days.astype(float) / float(day_count)

    payoff_table = pd.DataFrame({
        "Date_pricing": pricing_date,
        "Date_initiale": date_initiale,
        "S_0": S_0.values,                # <<< NEW COLUMN
        "S_ref": float(s_ref),
        "Date_sortie": exit_date.values,
        "Ratio_sortie_S_sur_Sref": ratio_sortie.values,
        "Nb_periodes": nb_periodes.values,
        "Payoff": payoff.values,
        "T_en_annees": T.values,
        "Called": called.values,
    }, index=scenarios)

    return payoff_table



In [ ]:
from typing import Tuple, Literal

Compounding = Literal["cc", "annual"]

def _parse_rate_to_float(x) -> float:
    """
    Accepts 0.0259, 2.59, '2,59%', '2.59%' ...
    Returns decimal rate (0.0259).
    """
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.floating)):
        # heuristic: if >1, treat as percent
        v = float(x)
        return v / 100.0 if v > 1.0 else v
    s = str(x).strip().replace("%", "").replace(",", ".")
    v = float(s)
    return v / 100.0 if v > 1.0 else v


def VM(
    payoff_table: pd.DataFrame,
    courbe_taux: pd.DataFrame,     # cột Year và Taux như ảnh
    col_year: str = "Year",
    col_taux: str = "Taux",
    compounding: Compounding = "annual",  # "annual" (mặc định) hoặc "cc"
) -> Tuple[pd.DataFrame, float]:
    """
    Tính PV và VM = mean(PV) theo courbe de taux (zéro rates theo maturity năm).

    - Nội suy tuyến tính r(T) theo Year.
    - Nếu compounding="annual": DF(T) = (1 + r(T))^{-T}
    - Nếu compounding="cc":     DF(T) = exp(-r(T)*T)
    """
    tbl = payoff_table.copy()

    curve = courbe_taux[[col_year, col_taux]].dropna().copy()
    curve[col_year] = curve[col_year].astype(float)
    curve[col_taux] = curve[col_taux].apply(_parse_rate_to_float).astype(float)
    curve = curve.sort_values(col_year)

    x = curve[col_year].to_numpy(dtype=float)
    y = curve[col_taux].to_numpy(dtype=float)

    T = tbl["T_en_annees"].astype(float).to_numpy()

    # extrapolation: clamp T ngoài range
    T_clip = np.clip(T, x.min(), x.max())
    rT = np.interp(T_clip, x, y)

    if compounding == "cc":
        df = np.exp(-rT * T)
    elif compounding == "annual":
        df = 1.0 / np.power(1.0 + rT, T)
    else:
        raise ValueError("compounding must be 'annual' or 'cc'")

    tbl["Taux_interp"] = rT
    tbl["Facteur_actualisation"] = df
    tbl["PV"] = tbl["Payoff"].astype(float).to_numpy() * df

    vm = float(tbl["PV"].mean())
    return tbl, vm


In [ ]:
path_decrement.to_excel("danhsach.xlsx", index=False)

In [ ]:
pay_tbl = Payoff(
    paths=path_decrement,
    s_ref=1000.0,
    barriere_sortie_anticipe=0.86,
    barriere_sortie_maturite=0.86,
    barriere_protection=0.60,
    premiere_annee_sortie=2,
    freq_sortie="trimestrielle",
    coupon_par_periode=0.019,
    annee_finale=12,
)



In [ ]:

# courbe_taux lấy từ sheet taux (Year, Taux)
tbl_vm, vm = VM(
    payoff_table=pay_tbl,
    courbe_taux=taux_df,
    col_year="Year",
    col_taux="Taux",
    compounding="annual"  # hoặc "cc" nếu bạn chắc là taux en continu
)

vm

999.9917880379736

In [ ]:
tbl_vm

,Date_pricing,Date_initiale,S_0,S_ref,Date_sortie,Ratio_sortie_S_sur_Sref,Nb_periodes,Payoff,T_en_annees,Called,Taux_interp,Facteur_actualisation,PV
scenario,,,,,,,,,,,,,
1,2025-12-31,2025-12-31,1000.0,1000.0,2027-12-31,0.971553,8.0,1152.0,2.000000,True,0.020930,0.959418,1105.250061
2,2025-12-31,2025-12-31,1000.0,1000.0,2027-12-31,1.374261,8.0,1152.0,2.000000,True,0.020930,0.959418,1105.250061
3,2025-12-31,2025-12-31,1000.0,1000.0,2027-12-31,1.271238,8.0,1152.0,2.000000,True,0.020930,0.959418,1105.250061
4,2025-12-31,2025-12-31,1000.0,1000.0,2028-12-31,0.967933,12.0,1228.0,3.002740,True,0.020931,0.939694,1153.944385
5,2025-12-31,2025-12-31,1000.0,1000.0,2028-03-31,0.983278,9.0,1171.0,2.249315,True,0.020930,0.954476,1117.691978
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9996,2025-12-31,2025-12-31,1000.0,1000.0,2027-12-31,1.826313,8.0,1152.0,2.000000,True,0.020930,0.959418,1105.250061
9997,2025-12-31,2025-12-31,1000.0,1000.0,2027-12-31,1.499593,8.0,1152.0,2.000000,True,0.020930,0.959418,1105.250061
9998,2025-12-31,2025-12-31,1000.0,1000.0,2027-12-31,1.075205,8.0,1152.0,2.000000,True,0.020930,0.959418,1105.250061


In [ ]:
vm

999.9917880379736

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# 0) CHECK: required objects exist
# ============================================================
required = ["decrement_paths", "Payoff", "VM", "taux_df"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Thiếu các object/hàm: {missing}. Hãy chạy cell định nghĩa/chứa chúng trước.")

# ============================================================
# 1) SETTINGS (bạn chỉnh đúng params của bạn ở đây)
# ============================================================


# ============================================================
# 2) BUILD TIME GRID (dùng lịch dates bạn đang dùng trong paths)
#    Bạn có thể thay `paths` bằng DataFrame nào có cột dates đúng timeline.
# ============================================================
dates = pd.to_datetime(paths.columns)   # <-- đảm bảo biến `paths` tồn tại (timeline)
dates = pd.DatetimeIndex(sorted(dates))

if len(dates) < 2:
    raise ValueError("Cần ít nhất 2 dates để simulate.")

times = np.array([(d - dates[0]).days / 365.0 for d in dates], float)
dt = np.diff(times)



# 4) VM_of_mu(mu): pipeline chuẩn của bạn
# ============================================================
import numpy as np
import pandas as pd

def make_VM_of_mu_monthly(
    start_date: str,
    n_months: int,
    n_sims: int,
    sigma_annual: float,
    s0_sim: float,
    decrement_value: float,
    decrement_mode: str,
    payoff_kwargs: dict,
    taux_df: pd.DataFrame,
    vm_kwargs: dict,
    seed: int = 42,
):
    """
    VM_of_mu(mu) đúng y hệt simulate_gbm_monthly:
    - dt = 1/12
    - dates = month-end
    - Z cố định để common random numbers
    """
    dt = 1.0 / 12.0
    dates = pd.date_range(start=pd.Timestamp(start_date), periods=n_months + 1, freq="M")

    # FIXED Z for ALL mu  (shape giống simulate_gbm_monthly: (n_months, n_sims))
    rng = np.random.default_rng(seed)
    Z = rng.standard_normal(size=(n_months, n_sims))

    def VM_of_mu(mu: float) -> float:
        # ---- simulate monthly GBM with drift=mu ----
        drift = (mu - 0.5 * sigma_annual**2) * dt
        diffusion = sigma_annual * np.sqrt(dt) * Z
        log_paths = np.vstack([np.zeros((1, n_sims)), np.cumsum(drift + diffusion, axis=0)])
        paths = s0_sim * np.exp(log_paths)

        paths_sim = pd.DataFrame(paths.T, index=np.arange(1, n_sims + 1), columns=dates)
        paths_sim.index.name = "scenario"

        # ---- decrement ----
        path_dec = decrement_paths(paths_sim, decrement_value=decrement_value, mode=decrement_mode)

        # ---- payoff ----
        pay_tbl = Payoff(paths=path_dec, **payoff_kwargs)

        # ---- discount / VM ----
        _, vm_val = VM(payoff_table=pay_tbl, courbe_taux=taux_df, **vm_kwargs)
        return float(vm_val)

    return VM_of_mu



# ============================================================
# 5) BISECTION SOLVER for mu
# ============================================================
def solve_mu_bisection(vm_func, target=1000.0, mu_low=-0.2, mu_high=0.5, tol=1e-2, max_iter=60):
    f_low = vm_func(mu_low) - target
    f_high = vm_func(mu_high) - target

    if f_low * f_high > 0:
        raise ValueError(
            "Không bracket được nghiệm. Hãy mở rộng mu_low/mu_high.\n"
            f"VM(mu_low)={vm_func(mu_low):.6f}, VM(mu_high)={vm_func(mu_high):.6f}, target={target}"
        )

    lo, hi = mu_low, mu_high
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        f_mid = vm_func(mid) - target
        if abs(f_mid) < tol:
            return mid
        if f_low * f_mid <= 0:
            hi = mid
            f_high = f_mid
        else:
            lo = mid
            f_low = f_mid
    return 0.5 * (lo + hi)





In [ ]:
START_DATE = "2025-12-31"
N_MONTHS = 145
N_SIMS = 10000
SIGMA = params.sigma_annual
S0_SIM = 1000.0

# các params SAF của bạn (như bạn đang chạy)
PAYOFF_KWARGS = dict(
    s_ref=1000.0,
    barriere_sortie_anticipe=0.86,
    barriere_sortie_maturite=0.86,
    barriere_protection=0.60,
    premiere_annee_sortie=2,
    freq_sortie="trimestrielle",
    coupon_par_periode=0.019,
    annee_finale=12,
)

# params curve
VM_KWARGS = dict(
    col_year="Year",
    col_taux="Taux",
    compounding="annual",
)

VM_of_mu = make_VM_of_mu_monthly(
    start_date=START_DATE,
    n_months=N_MONTHS,
    n_sims=N_SIMS,
    sigma_annual=SIGMA,
    s0_sim=S0_SIM,
    decrement_value=50.0,
    decrement_mode="cum",
    payoff_kwargs=PAYOFF_KWARGS,
    taux_df=taux_df,
    vm_kwargs=VM_KWARGS,
    seed=42,
)

print("VM(mu=0.0259) =", VM_of_mu(0.0259))
print("VM(mu=0.0000) =", VM_of_mu(0.0))

mu_star = solve_mu_bisection(VM_of_mu, target=1000.0, mu_low=-0.2, mu_high=0.5, tol=1e-2)
print("mu_star =", mu_star)
print("VM(mu_star) =", VM_of_mu(mu_star))


/tmp/ipython-input-2540299352.py:57: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(start=pd.Timestamp(start_date), periods=n_months + 1, freq="M")


VM(mu=0.0259) = 861.2310314768039
VM(mu=0.0000) = 750.437960107265
mu_star = 0.0691009521484375
VM(mu_star) = 999.9917880379736


In [ ]:
import numpy as np
import pandas as pd
from typing import Iterable, Dict, Any, Optional

def vm_table_for_s0_grid(
    s0_grid: Iterable[float],
    mu: float,
    sigma_annual: float,
    start_date: str,
    n_months: int,
    n_sims: int,
    decrement_value: float,
    decrement_mode: str,
    payoff_kwargs: Dict[str, Any],
    taux_df: pd.DataFrame,
    vm_kwargs: Dict[str, Any],
    seed: int = 42,
) -> pd.DataFrame:
    """
    Run a grid of S0 values -> compute VM for each S0 and return a comparison table.

    Pipeline for each S0:
      simulate_gbm_monthly(mu, sigma) with fixed Z
        -> decrement_paths(...)
        -> Payoff(...)
        -> VM(..., courbe_taux)
        -> VM = mean(PV)

    Uses common random numbers (fixed Z) so differences come from S0, not RNG noise.
    """
    dt = 1.0 / 12.0
    dates = pd.date_range(start=pd.Timestamp(start_date), periods=n_months + 1, freq="M")

    # fixed shocks for all S0 (CRN)
    rng = np.random.default_rng(seed)
    Z = rng.standard_normal(size=(n_months, n_sims))

    # precompute log increments that don't depend on S0
    drift = (mu - 0.5 * sigma_annual**2) * dt
    diffusion = sigma_annual * np.sqrt(dt) * Z
    log_paths = np.vstack([np.zeros((1, n_sims)), np.cumsum(drift + diffusion, axis=0)])  # (n_months+1, n_sims)
    exp_log_paths = np.exp(log_paths)  # reuse

    results = []
    for s0 in s0_grid:
        # simulate paths for this S0
        paths = float(s0) * exp_log_paths
        paths_sim = pd.DataFrame(paths.T, index=np.arange(1, n_sims + 1), columns=dates)
        paths_sim.index.name = "scenario"

        # decrement
        path_dec = decrement_paths(paths_sim, decrement_value=decrement_value, mode=decrement_mode)

        # payoff
        pay_tbl = Payoff(paths=path_dec, **payoff_kwargs)

        # discount / VM
        _, vm_val = VM(payoff_table=pay_tbl, courbe_taux=taux_df, **vm_kwargs)

        results.append({
            "S0_sim": float(s0),
            "VM": float(vm_val),
            "mu": float(mu),
            "sigma": float(sigma_annual),
            "n_sims": int(n_sims),
            "start_date": str(start_date),
            "n_months": int(n_months),
        })

    out = pd.DataFrame(results).sort_values("S0_sim").reset_index(drop=True)

    # optional: add deltas vs baseline S0=1000 if exists
    if (out["S0_sim"] == 1000.0).any():
        vm0 = float(out.loc[out["S0_sim"] == 1000.0, "VM"].iloc[0])
        out["VM_minus_VM_at_1000"] = out["VM"] - vm0
        out["VM_over_1000"] = out["VM"] / 1000.0

    return out


In [ ]:
s0_grid = [200,300,400,500,600,700,800, 900, 1000, 1100, 1200]

PAYOFF_KWARGS = dict(
    s_ref=1000.0,
    barriere_sortie_anticipe=0.86,
    barriere_sortie_maturite=0.86,
    barriere_protection=0.60,
    premiere_annee_sortie=2,
    freq_sortie="trimestrielle",
    coupon_par_periode=0.019,
    annee_finale=12,
)

VM_KWARGS = dict(
    col_year="Year",
    col_taux="Taux",
    compounding="annual",
)

table_vm = vm_table_for_s0_grid(
    s0_grid=s0_grid,
    mu=params.r_annual_cc,                 # hoặc mu bạn muốn test
    sigma_annual=params.sigma_annual,
    start_date="2025-12-31",
    n_months=145,
    n_sims=10000,
    decrement_value=50.0,
    decrement_mode="cum",
    payoff_kwargs=PAYOFF_KWARGS,
    taux_df=taux_df,
    vm_kwargs=VM_KWARGS,
    seed=42,
)

table_vm


/tmp/ipython-input-2681853349.py:32: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(start=pd.Timestamp(start_date), periods=n_months + 1, freq="M")


,S0_sim,VM,mu,sigma,n_sims,start_date,n_months,VM_minus_VM_at_1000,VM_over_1000
0,200.0,-70.523890,0.069101,0.255812,10000,2025-12-31,145,-1070.515678,-0.070524
1,300.0,141.795298,0.069101,0.255812,10000,2025-12-31,145,-858.196490,0.141795
2,400.0,337.162673,0.069101,0.255812,10000,2025-12-31,145,-662.829115,0.337163
3,500.0,516.382658,0.069101,0.255812,10000,2025-12-31,145,-483.609130,0.516383
4,600.0,671.000779,0.069101,0.255812,10000,2025-12-31,145,-328.991009,0.671001
5,700.0,787.424850,0.069101,0.255812,10000,2025-12-31,145,-212.566938,0.787425
6,800.0,884.107656,0.069101,0.255812,10000,2025-12-31,145,-115.884132,0.884108
7,900.0,951.307246,0.069101,0.255812,10000,2025-12-31,145,-48.684542,0.951307
8,1000.0,999.991788,0.069101,0.255812,10000,2025-12-31,145,0.000000,0.999992
9,1100.0,1032.928254,0.069101,0.255812,10000,2025-12-31,145,32.936466,1.032928


In [ ]:
with pd.ExcelWriter("output.xlsx", engine="openpyxl") as writer:
    paths.to_excel(writer, sheet_name="paths", index=False)
    path_decrement.to_excel(writer, sheet_name="path_decrement", index=False)